In [1]:
import pandas as pd
import mysql.connector
from pathlib import Path


In [ ]:
%pip install mysql-connector-python

In [15]:

# ==========================================================
# CONFIGURAÇÃO
# ==========================================================

CSV_PATH = Path(
    r"C:\Users\henri\Downloads\BASE_PERFIL_CLIENTES_EXEMPLO.csv"
)

MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": "",
    "database": "painel_gestao",
    "allow_local_infile": True
}

TABELA = "stg_perfil_clientes_raw"


In [16]:

# ==========================================================
# 1. VALIDAR ARQUIVO
# ==========================================================

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {CSV_PATH}"
    )


# ==========================================================
# 2. LER SOMENTE O CABEÇALHO
# ==========================================================

cabecalho = pd.read_csv(
    CSV_PATH,
    sep=";",
    encoding="cp1252",
    nrows=0
)

colunas = cabecalho.columns.tolist()

print(f"Quantidade de colunas: {len(colunas)}")
print(colunas)


Quantidade de colunas: 104
['JUNCAO_AGENCIA', 'CONTA', 'CNPJ_BASICO', 'CNPJ_COMPLETO', 'RAZAO_SOCIAL', 'NOME_GERENTE', 'FUNCAO', 'COD_CLIE', 'SUBSEGMENTO', 'SUBSEGMENTO_AJUSTADO', 'MAIOR_SUBSEGMENTO_AJUSTADO', 'DT_FUNDACAO', 'DT_ABERT_CTA', 'DULT_MOVTO_CTA', 'STATUS_CONTA', 'MUNICIPIO', 'UF', 'SITUACAO_RECEITA', 'SECAO_CNAE', 'DIVISAO_CNAE', 'GRUPO_CNAE', 'CLASS_CNAE', 'ATIVIDADE_CNAE', 'COD_CNAE', 'COD_NATUREZA', 'CEP', 'ENDERECO', 'RATING', 'GRAU_DE_RESTRICAO', 'PERFIL', 'PERFIL_RISCO', 'PORTE_EMPRESA', 'ANOS_FUNDACAO', 'ANOS_RELACIONAMENTO', 'ROB', 'FATURAMENTO_VUC', 'Indice Perfil Risco', 'AgConta', 'ANOMES', 'TOTAL_PRODUTOS', 'POSSUI_OPEN', 'POSSUI_ALELO', 'POSSUI_CIELO', 'POSSUI_CTA_GARANTIDA', 'POSSUI_CHEQUE', 'POSSUI_CDC', 'POSSUI_DESCONTO', 'POSSUI_CAGIRO', 'POSSUI_CREDITO', 'POSSUI_DEBITO_AUTOMATICO', 'POSSUI_CONSORCIO_PESADOS', 'POSSUI_CONSORCIO_IMOVEL', 'POSSUI_CONSORCIO_AUTO', 'POSSUI_PREVIDENCIA', 'POSSUI_SEGURO_SAUDE', 'POSSUI_SEGURO_AUTO', 'POSSUI_SEGURO_DENTAL', 'POSSU

In [17]:
import mysql.connector

MYSQL_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "etl_painel",
    "password": "EtL@Painel2026!",
    "database": "painel_gestao",
    "allow_local_infile": True
}

conn = mysql.connector.connect(**MYSQL_CONFIG)

print("Conectado ao MySQL com sucesso!")

cursor = conn.cursor()
cursor.execute("""
    SELECT VERSION(), DATABASE(), @@port
""")

print(cursor.fetchone())

cursor.close()
conn.close()

Conectado ao MySQL com sucesso!
('8.0.44', 'painel_gestao', 3306)


In [18]:

# ============================================================
# 3. CONECTAR MYSQL
# ============================================================

conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

print("\nConectado ao MySQL.")


Conectado ao MySQL.


In [22]:

# ============================================================
# 4. APAGAR STAGE ANTERIOR
# ============================================================

cursor.execute("""
    DROP TABLE IF EXISTS stg_perfil_clientes_raw
""")

print("4. Stage anterior removida ✅")


4. Stage anterior removida ✅


In [23]:

# ============================================================
# 5. CRIAR STAGE DINAMICAMENTE
# ============================================================

colunas_sql = ",\n".join(
    f"`{str(col).replace('`', '')}` TEXT NULL"
    for col in colunas
)

sql_create = f"""
CREATE TABLE stg_perfil_clientes_raw (

    id_carga BIGINT UNSIGNED
        NOT NULL AUTO_INCREMENT,

    {colunas_sql},

    dt_carga DATETIME
        NOT NULL DEFAULT CURRENT_TIMESTAMP,

    PRIMARY KEY (id_carga)

)
ENGINE=InnoDB
ROW_FORMAT=DYNAMIC
DEFAULT CHARSET=utf8mb4
COLLATE=utf8mb4_unicode_ci;
"""

cursor.execute(sql_create)
conn.commit()

print("5. Stage criada ✅")



5. Stage criada ✅


In [24]:

# ============================================================
# 6. PREPARAR LOAD DATA
# ============================================================

lista_colunas = ",\n".join(
    f"`{str(col).replace('`', '')}`"
    for col in colunas
)

caminho_csv = CSV_PATH.as_posix()

sql_load = f"""
LOAD DATA LOCAL INFILE '{caminho_csv}'
INTO TABLE stg_perfil_clientes_raw
CHARACTER SET latin1

FIELDS TERMINATED BY ';'
OPTIONALLY ENCLOSED BY '"'

LINES TERMINATED BY '\\r\\n'

IGNORE 1 LINES

(
    {lista_colunas}
);
"""

In [25]:

# ============================================================
# 7. CARREGAR
# ============================================================

print("6. Iniciando carga...")

cursor.execute(sql_load)
conn.commit()

print("7. CSV carregado ✅")


# ============================================================
# 8. VALIDAR QUANTIDADE
# ============================================================

cursor.execute("""
    SELECT COUNT(*)
    FROM stg_perfil_clientes_raw
""")

qtd = cursor.fetchone()[0]

print(f"8. Registros carregados: {qtd}")


6. Iniciando carga...
7. CSV carregado ✅
8. Registros carregados: 30


In [26]:

# ============================================================
# 9. CONFERIR PRIMEIRAS LINHAS
# ============================================================

cursor.execute("""
    SELECT
        id_carga,
        CONTA,
        CNPJ_COMPLETO,
        RAZAO_SOCIAL,
        ROB,
        FATURAMENTO_VUC,
        ANOMES,
        dt_carga
    FROM stg_perfil_clientes_raw
    LIMIT 5
""")

for linha in cursor.fetchall():
    print(linha)


(1, '100000', '12000000100010', 'EMPRESA EXEMPLO 01 LTDA', '30224.47', '5144260.24', '202609', datetime.datetime(2026, 9, 10, 20, 48, 1))
(2, '100001', '12000001100111', 'EMPRESA EXEMPLO 02 LTDA', '215498.57', '7409098.14', '202609', datetime.datetime(2026, 9, 10, 20, 48, 1))
(3, '100002', '12000002100212', 'EMPRESA EXEMPLO 03 LTDA', '140356.58', '4351881.92', '202609', datetime.datetime(2026, 9, 10, 20, 48, 1))
(4, '100003', '12000003100313', 'EMPRESA EXEMPLO 04 LTDA', '145133.76', '5255367.47', '202609', datetime.datetime(2026, 9, 10, 20, 48, 1))
(5, '100004', '12000004100414', 'EMPRESA EXEMPLO 05 LTDA', '159172.68', '4220439.11', '202609', datetime.datetime(2026, 9, 10, 20, 48, 1))
